In [1]:
%pip install medmnist
%pip install mlxtend
%pip install torchmetrics

  Using cached medmnist-3.0.2-py3-none-any.whl.metadata (14 kB)
  Using cached numpy-2.4.3-cp313-cp313-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.1-cp313-cp313-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached scikit_learn-1.8.0-cp313-cp313-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached scikit_image-0.26.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (15 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached pillow-12.1.1-cp313-cp313-macosx_11_0_arm64.whl.metadata (8.8 kB)
  Using cached fire-0.7.1-py3-none-any.whl.metadata (5.8 kB)
  Using cached torch-2.11.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (29 kB)
  Using cached torchvision-0.26.0-cp313-cp313-macosx_12_0_arm64.whl.metadata (5.5 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached scipy-1.17.1-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached imageio-2.37.3

In [2]:
import torch.utils.data as data
import pandas as pd
from torchvision import transforms
from torch.utils.data import random_split
import torch

# DeepGA is an algorithm for evolve a convolutional neural network.


In [4]:
from DeepGA.Operators import *
from DeepGA.EncodingClass import Encoding
from DeepGA.Decoding import *
from DeepGA.DataReader import *
from DeepGA.DistributedTraining import *
from DeepGA.DeepGA import *

In [5]:
root_dir = '/Users/mauricio/Documents/Ecuela-UV/Semestre-8/Servicio/DeepGA/clasificacion de radiografias/radiografiamas'

# Data reader

In [6]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np
from torchvision.transforms import ToTensor

class CustomImageDataset(Dataset):
    def __init__(self, dataframe, root_dir, label_col='mes_scoring_0_3', img_col='img_url', transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.root_dir = root_dir
        self.label_col = label_col
        self.img_col = img_col
        self.transform = transform
        # Map string labels to integers deterministically
        unique_labels = sorted(self.dataframe[self.label_col].astype(str).unique())
        self.label_map = {label: idx for idx, label in enumerate(unique_labels)}

    def __len__(self):
        return len(self.dataframe)

    def _find_image_path(self, filename):
        # Normalize and remove extension
        base = os.path.splitext(os.path.basename(str(filename)).strip())[0]
        # fallback to common extensions
        for ext in ('.jpg', '.jpeg', '.png', '.bmp', '.tiff'):
            candidate = os.path.join(self.root_dir, base + ext)
            if os.path.isfile(candidate):
                return candidate
        return None

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        raw_fname = row[self.img_col]
        img_path = self._find_image_path(raw_fname)
        if img_path is None:
            raise FileNotFoundError(f"Image not found for index {idx}: '{raw_fname}' (searched in '{self.root_dir}')")
        try:
            pil_image = Image.open(img_path).convert('RGB')
        except Exception as e:
            raise RuntimeError(f"Failed to open image '{img_path}': {e}")

        if self.transform is not None:
            image = self.transform(pil_image)
        else:
            # Default transform to tensor if none provided
            image = ToTensor()(pil_image)

        label_raw = row[self.label_col]
        label_key = str(label_raw)
        if label_key not in self.label_map:
            raise KeyError(f"Label '{label_raw}' not found in label_map for index {idx}")
        label = int(self.label_map[label_key])

        return image, label

def validate_dataset(dataset, n_samples=5):
    """Carga hasta n_samples del dataset y muestra formas y etiquetas únicas para validar."""
    n = min(len(dataset), n_samples)
    imgs = []
    labels = []
    for i in range(n):
        img, lbl = dataset[i]  # puede lanzar excepción si hay problemas
        imgs.append(img)
        labels.append(lbl)
    shapes = [tuple(img.shape) for img in imgs]
    print(f"Validación: cargadas {n} muestras. Formas: {shapes}. Etiquetas: {labels}. Label map: {dataset.label_map}")
    return shapes, labels


def loading_data():
    # Cargar dos columnas específicas
    df = pd.read_excel('dataframe_complete_GT.xlsx', usecols=['img_url', 'mes_scoring_0_3'])
    transform = transforms.Compose([transforms.Resize((128, 128)), transforms.ToTensor()])
    dataset = CustomImageDataset(df, root_dir=root_dir, transform=transform)

    total_len = len(dataset)
    train_len = int(0.8 * total_len)
    val_len = total_len - train_len

    # Reproducible random split
    gen = torch.Generator().manual_seed(42)
    train_dataset, val_dataset = random_split(dataset, [train_len, val_len], generator=gen)

    BATCH_SIZE = 64
    
    #train_dl = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    #val_dl = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    total_dl = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    #print(f"Dataset sizes -> total: {total_len}, train: {train_len}, val: {val_len}")
    n_channels = 3    
    return total_dl, total_dl, n_channels, len(dataset.label_map), 128

### Call DeepGA for evolve CNN´s

In [40]:
'''Defining DeepGA hyperparameters'''

#Convolutional layers
FSIZES = [3, 5, 7, 9] # Odd Sizes Are Preferred
NFILTERS = [8, 16, 32]

#FSIZES = [9, 11, 15, 17, 19, 21] # Odd Sizes Are Preferred
#NFILTERS = [4, 8, 16]

#Pooling layers
PSIZES = [2,3] #[2,3,4,5]
PTYPE = ['max', 'avg']

#Fully connected layers
NEURONS = [16, 32, 64, 128] # for big layers

#Defining learning rate
lr = 1e-4

#Maximun and minimum numbers of layers to initialize networks
min_conv = 2 # 30
max_conv = 8 # 60
min_full = 1
max_full = 3 # 10
max_params = 5e6
train_epochs = 5 # Epochs to train the best individual found by the GA

'''Genetic Algorithm Parameters'''
cr = 0.7   # Crossover rate
mr = 0.5   # Mutation rate
N = 20      # Population size 20 Se mantuvo en 20
T = 30      # Number of generations
t_size = 5 # tournament size 5
w = 0.1    # penalization weight   0.3
#chck_dir = '/content/drive/MyDrive/checkpoint/'  # Root folder for dumping/loading pickle
chck_dir = 'point/'  # Root folder for dumping/loading pickle
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_dl, val_dl, n_channels, n_classes, out_size = loading_data() #Loading data

loss_func = nn.CrossEntropyLoss()
execution_ID = 12810 #Execution ID for checkpoint, change it for a new execution
results, pop, bestind  = deepGA(execution_ID, True, train_epochs = train_epochs, train_dl=train_dl, val_dl=val_dl,  lr=lr,
                       min_conv=min_conv, max_conv=max_conv, min_full=min_full, max_full=max_full, max_params=max_params,
                       cr=cr, mr=mr, N=N, T=T, t_size=t_size, w=w, device=device, chck_dir=chck_dir,
                       n_channels =  n_channels , n_classes=n_classes, out_size = out_size, loss_func=loss_func)

# training to more epochs

finalEpochs = 250
CNNModel = final_evaluation(execution_ID, bestind, train_dl, val_dl, lr, max_params, w, device, finalEpochs, loss_func, chck_dir, n_channels =  n_channels , n_classes=n_classes, out_size = out_size)

Re-Initialize population
The maximum number of generations has been reached. Please run a new execution.
--------------------------------------------
    Accuracy   Fitness  No. Params   MeanFit   MeanAcc    MeanPar
0   0.790010  0.777741     1663396  0.705989  0.725178  2333570.0
1   0.790010  0.777741     1663396  0.729746  0.745719  2070051.2
2   0.790010  0.777741     1663396  0.745903  0.758155  1821832.8
3   0.853211  0.810170     2886020  0.756214  0.757594  1281064.8
4   0.853211  0.810170     2886020  0.764852  0.769164  1369786.0
5   0.880734  0.820801     3592972  0.773289  0.783078  1574064.0
6   0.916412  0.855288     3474124  0.782238  0.799643  1872042.0
7   0.916412  0.855288     3474124  0.792299  0.817023  2151115.6
8   0.916412  0.855288     3474124  0.803386  0.839857  2624264.8
9   0.916412  0.855288     3474124  0.819331  0.866310  3017413.6
10  0.916412  0.855288     3474124  0.833825  0.884404  3106904.4
11  0.916412  0.855288     3474124  0.842800  0.893170  30

### Building  model

In [41]:
# Evaluate images listed in the Excel file using CNNModel and save predictions (including ground truth)
if 'CNNModel' not in globals():
    raise RuntimeError("CNNModel not found in globals()")

# Read excel (same columns used in loading_data)
df = pd.read_excel('dataframe_complete_GT.xlsx', usecols=['img_url', 'mes_scoring_0_3'])
#root_dir = 'D:/Endo-UC_TrainSet-I/Endo-UC_TrainSet-I/image_folder/'

# Use same transform as training (out_size expected to exist)
transform = transforms.Compose([transforms.Resize((out_size, out_size)), transforms.ToTensor()])

dataset = CustomImageDataset(df, root_dir=root_dir, label_col='mes_scoring_0_3', img_col='img_url', transform=transform)

# Prepare model
model = CNNModel
model.eval()
model.to(device)

# inverse label map
inv_label_map = {v: k for k, v in dataset.label_map.items()}

batch_size = 32
imgs_batch = []
names_batch = []
labels_batch = []
preds_names = []
preds_idx = []
preds_img_urls = []
preds_gt = []

for i in range(len(dataset)):
    try:
        img, _ = dataset[i]
    except Exception as e:
        print(f"Skipping index {i}: {e}")
        continue

    imgs_batch.append(img.unsqueeze(0))
    img_url = str(dataset.dataframe.iloc[i][dataset.img_col])
    gt_label = dataset.dataframe.iloc[i][dataset.label_col]
    names_batch.append(img_url)
    labels_batch.append(gt_label)

    if len(imgs_batch) >= batch_size or i == len(dataset) - 1:
        batch_tensor = torch.cat(imgs_batch, dim=0).to(device)
        with torch.no_grad():
            outputs = model(batch_tensor) # Get model outputs   
        if isinstance(outputs, (tuple, list)):
            outputs = outputs[0] #  Handle models that return multiple outputs  
        preds = outputs.argmax(dim=1).cpu().numpy() # Get predicted class indices
        for p, nm, gt in zip(preds, names_batch, labels_batch): #   Record predictions  
            preds_idx.append(int(p))
            preds_names.append(inv_label_map.get(int(p), str(int(p))))
            preds_img_urls.append(nm)
            preds_gt.append(gt)
        imgs_batch = []
        names_batch = []
        labels_batch = []

# Build final DataFrame aligned to recorded predictions (including original mes_scoring_0_3)
final_df = pd.DataFrame({
    'img_url': preds_img_urls,
    'mes_scoring_0_3': preds_gt,
    'predicted_class_idx': preds_idx,
    'predicted_class_label': preds_names
})

final_df.to_csv('predictions_128_250.csv', index=False)
print(final_df.head())

                                            img_url mes_scoring_0_3  \
0  image_folder/image_clhhyjzdx00os07xbernefuly.jpg           MES-1   
1  image_folder/image_clhhyjzoj03mw07tl9vpdgbec.jpg           MES-1   
2  image_folder/image_clhhyscmn00uy07z726k8ftwr.jpg           MES-1   
3  image_folder/image_clhhz42m2044k07wj5cnme69t.jpg           MES-0   
4  image_folder/image_clhhzk82g02pw07wc1y018vi4.jpg           MES-0   

   predicted_class_idx predicted_class_label  
0                    1                 MES-1  
1                    1                 MES-1  
2                    1                 MES-1  
3                    0                 MES-0  
4                    0                 MES-0  


In [42]:
def extract_features(feature_extractor, loader):
    features = []
    labels = []
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    feature_extractor = feature_extractor.to(device)
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            outputs = feature_extractor(images).squeeze()  # Remove singleton dimensions
            features.append(outputs.cpu().numpy())
            labels.append(targets.numpy())
    features = np.concatenate(features, axis=0)
    labels = np.concatenate(labels, axis=0)
    return features, labels

def predictions(feature_extractor_CNN_Model, LDA_model, data):
    features, labels_test = extract_features(feature_extractor_CNN_Model, data) # Getting features of test images
    predicted_labels = LDA_model.predict(features)
    probabilities = LDA_model.predict_proba(features)
    reduced_features = LDA_model.transform(features) # LDA reduction of test features, are projected onto a lower-dimensional space
    return probabilities, predicted_labels, reduced_features, labels_test



In [1]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

In [44]:
CNNModel.eval()
DeepGA_CNN_Model = copy.deepcopy(CNNModel)
DeepGA_CNN_Model.eval()  # Set the model to evaluation mode
# Remove the final classification layer to get features
DeepGA_CNN_Model.classifier = nn.Sequential(*list(DeepGA_CNN_Model.classifier.children())[:-1]) 
#DeepGA_CNN_Model.classifier = nn.Sequential(*list(DeepGA_CNN_Model.classifier.children())[:-2]) 
feature_extractor_CNN_Model = DeepGA_CNN_Model
feature_extractor_CNN_Model.eval()
train_dl, val_dl, n_channels, n_classes, out_size = loading_data() #Loading data
features_train, labels_train = extract_features(feature_extractor_CNN_Model, train_dl) # Getting features of train images   
#LDA Classifier
LDA_model = LinearDiscriminantAnalysis(solver='eigen', n_components=3)
LDA_model.fit(features_train, labels_train)


LinAlgError: The leading minor of order 11 of B is not positive definite. The factorization of B could not be completed and no eigenvalues or eigenvectors were computed.

In [37]:
CNNModel

CNN(
  (extraction): Sequential(
    (0): Conv2d(3, 8, kernel_size=(7, 7), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (4): Conv2d(8, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): Conv2d(16, 32, kernel_size=(7, 7), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
  )
  (classifier): Sequential(
    (0): Linear(in_features=107648, out_features=32, bias=True)
    (1): ReLU(inplace=True)
    (2): Linear(in_features=32, out_features=64, bias=True)
    (3): ReLU(inplace=True)
    (4): Linear(in_features=64, out_features=4, bias=True)
  )
)

### Saving model

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import onnx
import onnxruntime as ort
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
from pathlib import Path
import copy
import zipfile
import joblib

# Prepare output dir for DeepGA andONNX models
out_dir = Path('models/ensemble')
out_dir.mkdir(parents=True, exist_ok=True)

export_model = copy.deepcopy(feature_extractor_CNN_Model)

#dummy_shape = infer_dummy_shape(export_model)
wrapped = export_model

wrapped.eval()
wrapped.to('cpu')
#dummy_input = torch.randn(*dummy_shape)

cnn_joblib_path = out_dir / 'DeepGA.joblib'
joblib.dump(wrapped, cnn_joblib_path)
print('Saved DeepGA Model to', wrapped)


if 'LDA_model' not in globals():
    raise RuntimeError('LDA_model not found')


lda_model_path = out_dir / 'lda.joblib'

if 'LDA_model' in globals():
    joblib.dump(LDA_model, lda_model_path)
    print('Saved LDA model to', lda_model_path)
else:
    print('No LDA model found; skipping')

gmm_path = out_dir / 'gmm.joblib'
if 'gmm' in globals():
    joblib.dump(gmm, gmm_path)
    print('Saved GMM to', gmm_path)
else:
    print('No GMM found; skipping')

zip_path = out_dir.parent / 'ensemble.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    if cnn_joblib_path.exists():
        zf.write(cnn_joblib_path, arcname='DeepGA.joblib')
    if lda_model_path.exists():
        zf.write(lda_model_path, arcname='lda.jolib')
    if gmm_path.exists():
        zf.write(gmm_path, arcname='gmm.joblib')
print('Created', zip_path)

print('Export and packaging complete')